In [58]:
import requests
from bs4 import BeautifulSoup
import re
import jwt
import uuid
import time
from cryptography.hazmat.primitives.asymmetric import ec
import base64

In [59]:
search_term = "professional2"

In [60]:
# what is sent in the request body for the search POST
data_raw = """{
   "userId":"",
   "config":{
      "responseToggles":[
         "QUERY_SUGGESTION_WEB_1"
      ]
   },
   "pageSize":120,
   "pageToken":"",
   "searchSessionId":"c83bbb3d087c6baa398a460d06a0cefc",
   "source":"BaseSerp",
   "indexRouting":"INDEX_ROUTING_UNSPECIFIED",
   "thumbnailTypes":[
      
   ],
   "searchCondition":{
      "keyword":"professional2",
      "excludeKeyword":"",
      "sort":"SORT_CREATED_TIME",
      "order":"ORDER_DESC",
      "status":[
         
      ],
      "sizeId":[
         
      ],
      "categoryId":[
         
      ],
      "brandId":[
         
      ],
      "sellerId":[
         
      ],
      "priceMin":0,
      "priceMax":0,
      "itemConditionId":[
         
      ],
      "shippingPayerId":[
         
      ],
      "shippingFromArea":[
         
      ],
      "shippingMethod":[
         
      ],
      "colorId":[
         
      ],
      "hasCoupon":false,
      "attributes":[
         
      ],
      "itemTypes":[
         
      ],
      "skuIds":[
         
      ],
      "shopIds":[
         
      ],
      "excludeShippingMethodIds":[
         
      ]
   },
   "serviceFrom":"suruga",
   "withItemBrand":true,
   "withItemSize":false,
   "withItemPromotions":true,
   "withItemSizes":true,
   "withShopname":false,
   "useDynamicAttribute":true,
   "withSuggestedItems":true,
   "withOfferPricePromotion":true,
   "withProductSuggest":true,
   "withParentProducts":false,
   "withProductArticles":true,
   "withSearchConditionId":false,
   "withAuction":true,
   "laplaceDeviceUuid":"fa3a3ca3-e696-4d01-a2f5-49eb502e1c38"
}"""

In [61]:
mercari_url = "https://api.mercari.jp/v2/entities:search"

so the idea is that we include our public key as part of the token, that's what the `alg` and `jwk` are for. my understanding is that the JWK (JSON Web Key) basically contains all the relevant information for whatever encryption algorithm you're using. mercari uses elliptic curve so we too use elliptic curve, and so in `headers` we have `alg` set to `"ES256"`, and then in JWK, where we specify all the things we need to for our particular encryption method, we say "here is the x and y", as the x/y is all a public key is for elliptic curve. ok!

also, base64url gets used instead of base64 because base64 will include '+' and '\' characters, which will not play nice with the fact that this is being sent in an HTTP header. 

i'm not sure how much of this process was truly necessary, since on mercari you view results without being logged in, and hence this token has no real meaning. but at the same time, if you try to make a GET/POST without including a valid one in `DPoP` of the header, you get 401'd.

In [62]:
# generate a key pair once and reuse it across requests
PRIVATE_KEY = ec.generate_private_key(ec.SECP256R1())
PUBLIC_KEY = PRIVATE_KEY.public_key()

def int_to_base64url(n:int, length:int=32) -> str:
    # all decode does is convert this from bytes to a utf-8 string
    # even though this is base64url the python function leaves in the padding '=' chars so remove them
    # length = 32 bytes because we're using, well, a 256 bit key
    return base64.urlsafe_b64encode(n.to_bytes(length, byteorder="big")).decode().rstrip("=")

# thank you https://www.jwt.io/
JWK = {
    "crv": "P-256",
    "kty": "EC",
    "x": int_to_base64url(PUBLIC_KEY.public_numbers().x),
    "y": int_to_base64url(PUBLIC_KEY.public_numbers().y),
}

# the only thing that actually needs to change between requests is the payload
def generate_dpop_proof(method: str, url: str) -> str:
    headers = {
        "typ": "dpop+jwt",
        "alg": "ES256",
        "jwk": JWK,
    }
    payload = {
        "jti": str(uuid.uuid4()),   # Unique ID per request
        "htm": method,              # HTTP method
        "htu": url,                  # Target URL
        "iat": int(time.time()),     # Current timestamp
    }
    # this does the "signature" part of the header.payload.signature format
    return jwt.encode(
        payload,
        PRIVATE_KEY,
        algorithm="ES256",
        headers=headers
    )

In [63]:
mercari_headers = {
  'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:141.0) Gecko/20100101 Firefox/141.0',
  'Accept': 'application/json, text/plain, */*',
  'Accept-Language': 'ja',
  'Accept-Encoding': 'gzip, deflate, br, zstd',
  'Content-Type': 'application/json',
  'X-Platform': 'web',
  'X-Country-Code': 'US',
  'Origin': 'https://jp.mercari.com',
  'Connection': 'keep-alive',
  'Referer': 'https://jp.mercari.com/',
  'Sec-Fetch-Dest': 'empty',
  'Sec-Fetch-Mode': 'cors',
  'Sec-Fetch-Site': 'cross-site',
  'Idempotency-Key': '"11432578277613167492"',
  'Pragma': 'no-cache',
  'Cache-Control': 'no-cache',
  'TE': 'trailers',
}

This is where we actually get the listings for the search back, get the 120 results that would appear when searching for the term we're looking for by "Newest".

In [ ]:
# get the conversion rate since shops pages just don't send USD prices
currency_url = 'https://api.mercari.jp/v2/getCurrencyConversionRate/currency'
mercari_headers.update({'DPoP': generate_dpop_proof("GET", currency_url)})
jpy_to_usd_response = requests.get(currency_url, params={'currency_code' : 'USD'}, headers=mercari_headers)
print(jpy_to_usd_response)

JPY_TO_USD = jpy_to_usd_response.json()['rate']
print(JPY_TO_USD)


mercari_headers.update({'DPoP': generate_dpop_proof("POST", mercari_url)})
api_response = requests.post(mercari_url, data=data_raw, headers=mercari_headers)
print(api_response)

jason = api_response.json()
print(jason)

# I'm not sure exactly when the id isn't exactly 11 chars long, since most seem to be,
#  but I've seen shorter ones at 9 digits, so I'm just going to be safe-ish and go with a range of 9-13 chars
item_pattern = re.compile("m[0-9]{9,13}")
shop_items = []
user_items = []

# store a dict for each listing; this is being created now because the price in dollars of all items is listed
# on this 'listings' page, but the price is only shown in JPY on items sold by shops (for some reason)
# TODO: for whenever this is more, like, actually an application, store some globabl set(?) of 'seen' ids
listings = []

for item in jason['items']:
    item_id = item['id']
    if item_pattern.match(item_id):
        user_items.append(item_id)
    else:
        shop_items.append(item_id)

<Response [200]>
0.00668
<Response [200]>
{'meta': {'nextPageToken': 'v1:1', 'previousPageToken': '', 'numFound': '218', 'properties': []}, 'items': [{'id': 'm62787078165', 'sellerId': '556531251', 'buyerId': '', 'status': 'ITEM_STATUS_ON_SALE', 'name': 'HHKB Professional2 TypeS PD-KB400WS US配列', 'price': '18000', 'created': '1777960075', 'updated': '1777960075', 'thumbnails': ['https://static.mercdn.net/thumb/item/webp/m62787078165_1.jpg?1777960075'], 'itemType': 'ITEM_TYPE_MERCARI', 'itemConditionId': '3', 'shippingPayerId': '2', 'itemSizes': [], 'itemBrand': None, 'itemPromotions': [], 'shopName': '', 'itemSize': None, 'shippingMethodId': '14', 'categoryId': '3710', 'isNoPrice': False, 'title': '', 'isLiked': False, 'photos': [{'uri': 'https://static.mercdn.net/item/detail/webp/photos/m62787078165_1.jpg?1777960075'}], 'auction': None, 'shop': None}, {'id': 'm83754167965', 'sellerId': '533995526', 'buyerId': '', 'status': 'ITEM_STATUS_ON_SALE', 'name': 'HHKB Professional2 TypeS PD-KB

In [65]:
shop_items

['2JQdgew77i8sJJKegMQnpG',
 '2JQ6278E36PEhU2wEvmCnp',
 'LMvu2vebofFhotVzXBsotG',
 '2JPCX2ccNoXiVGmpGTwqCg',
 'LbX5G5kW9aRSmpKmd2VLaD',
 'JwSFRTBPA3GMCyMfa9efwh',
 '2JMgBnSoDZqzr6WJi6iApi',
 '2JM3M4UFzqaFwKwwq2e6CT',
 '2JLzum4uGce2vZojUPR8HD',
 '2JLzukzA74sxejMgMfYHRP',
 '2JJ5ydJg9qXPjDTGJuwY5a',
 '2JJ5yqd3esLYf5mEtq5sBN',
 '5JtSHaHJszfiwSPb3RhXy5',
 'iGAoCbjHn3R9zZj3XtwZAZ',
 'tqQa28CW9frwGhhYUZhZab',
 'PM3mEa7Q4pkBGAVmP8ySYD',
 '2JGXzsFeVU4QvVFuBpVNAB',
 '2JGXzsBbFwtZoqCyLu3P7N',
 'MkGqL78K3hfSMmW2M7Qenm']

In [66]:
user_items

['m62787078165',
 'm83754167965',
 'm16359558532',
 'm34858460268',
 'm34263027785',
 'm86745571889',
 'm91139152159',
 'm96513269270',
 'm36484230456',
 'm37758880641',
 'm98495088673',
 'm78197104566',
 'm27605696926',
 'm68328763495',
 'm31021897996',
 'm37067855704',
 'm13820752420',
 'm23418420044',
 'm70787746904',
 'm95711421897',
 'm73055841321',
 'm79058476742',
 'm19506562005',
 'm12319226236',
 'm12220775819',
 'm32017279483',
 'm82177297713',
 'm16723979637',
 'm81848983941',
 'm24013871573',
 'm29114415282',
 'm34598828274',
 'm62698657442',
 'm53504768264',
 'm51252395099',
 'm88868047096',
 'm80167241376',
 'm21258041595',
 'm72528521140',
 'm58456175866',
 'm75372196071',
 'm67387214819',
 'm40575818410',
 'm89954714628',
 'm78702369053',
 'm26970363320',
 'm87354086076',
 'm85771342750',
 'm48759340790',
 'm89237416543',
 'm89248321323',
 'm21650504182',
 'm29008316107',
 'm83675564489',
 'm85499079963',
 'm55314358919',
 'm82798076251',
 'm79618857974',
 'm43409499319

Test getting a information on a particular item back (i.e., getting back information like a description that's only shown on that particular item's page).

In [67]:
shop_url_please_god = "https://api.mercari.jp/v1/marketplaces/shops/products/2JQdgew77i8sJJKegMQnpG?view=FULL"
mercari_headers.update({'DPoP': generate_dpop_proof("GET", shop_url_please_god)})
item_response = requests.get(shop_url_please_god, headers = mercari_headers)

In [68]:
item_response

<Response [200]>

In [69]:
item_response_json = item_response.json()

In [70]:
item_response_json['productDetail']['description']

'HHKB Professional2です。\n若干使用感はありますが目立つ汚れはありません。\n全てのキーが反応することも確認済みです。\n箱やケーブル意外の付属品はありません。写真に写っているものが全てとなります。\n\n中古、自宅保管品ですので、ご理解頂ける方のみどうぞ宜しくお願い致します。\n\n■状態、画像、採寸について\n＊出品商品は1点物の中古品がほとんどのため商品状態も様々です。可能な限り細かく状態を記載しておりますが記載しきれない場合もございます。デリケートな方は購入をお見送り下さい。\n\n＊素人撮影のため実物と画像で色が異なって見える場合がございますのでご了承ください。\nまたお使いのパソコンモニター・スマホの色調でも違って見える場合もございますのでご理解のほどよろしくお願いいたします。\n\n＊記載寸法は手作業で採寸しているため実際にお届けする商品と多少の誤差がある場合がございますのであらかじめご了承くださいませ。\n\n↓他にも様々な商品を出品してるので是非ご覧下さい！'

In [71]:
my_rad_id = shop_items[1]
format_test = "https://api.mercari.jp/v1/marketplaces/shops/products/{0}?view=FULL".format(my_rad_id)

mercari_headers.update({'DPoP': generate_dpop_proof("GET", format_test)})
item_response = requests.get(format_test, headers = mercari_headers)
print(item_response)

<Response [200]>


In [72]:
def get_listing_json_from_id(id:str):
    headers = {
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:141.0) Gecko/20100101 Firefox/141.0',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'ja',
    'Accept-Encoding': 'gzip, deflate, br, zstd',
    'Content-Type': 'application/json',
    'X-Platform': 'web',
    'X-Country-Code': 'US',
    'Origin': 'https://jp.mercari.com',
    'Connection': 'keep-alive',
    'Referer': 'https://jp.mercari.com/',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'cross-site',
    'Idempotency-Key': '"11432578277613167492"',
    'Pragma': 'no-cache',
    'Cache-Control': 'no-cache',
    'TE': 'trailers',
    }
    item_pattern = re.compile("m[0-9]{9,13}")
    if item_pattern.match(id):
        # i don't know what their engineers are cooking up here
        url = "https://api.mercari.jp/items/get?id={0}&include_item_attributes=true&include_product_page_component=true&include_non_ui_item_attributes=true&include_donation=true&include_item_attributes_sections=true&include_auction=true".format(id)
        item_type = "user"
    else:
        # i guess 'shops' is a newer feature so they got the tech-debtless API
        url = "https://api.mercari.jp/v1/marketplaces/shops/products/{0}?view=FULL".format(id)
        item_type = "shop"

    headers.update({'DPoP': generate_dpop_proof("GET", url)})
    response = requests.get(url, headers=headers)
    print(response)
    if not response.ok:
        return None

    jason = response.json()
    if item_type == "user":
        return {
            "id": id,
            "title": 2,
            "price_usd" : 
        }
    else:
        return {

        }

SyntaxError: expression expected after dictionary key and ':' (3397968362.py, line 42)

In [ ]:
some_id = user_items[0]
print(some_id)
testing = get_listing_json_from_id(some_id)
print(testing)
testing = get_listing_json_from_id(shop_items[0])

NameError: name 'user_items' is not defined

In [ ]:
listings = []
for idx in range(10):
    jason = get_listing_json_from_id(user_items[idx])
    if jason != None:
        listings.append(jason)
        print(jason['data']['description'])
    else:
        continue

<Response [200]>
HHKB Professional2の英字配列、墨モデルになります。
合わせてUSBタイプA to miniBの3mケーブルをお付けします。
使用期間は2年ほどで、表裏に少々キズがあります。
梱包の際は別商品の箱に詰めて発送いたしますのでご了承下さい。
<Response [200]>
SLIK PROFESSIONAL 2 大型三脚 スリック プロフェッショナル 2 カメラ用三脚 ブラック

型番: PROFESSIONAL2
製造番号: 6557
サイズ(幅/奥行き/高さ): 約 53-128/53-128/77-184 cm
重量: 約4.7kg
※素人寸法です。若干の誤差はご了承下さい。
本体のみでその他の物は付属いたしません。

全体的に使用感があり、脚やレバーに傷や錆、劣化がありポロポロしておりますが、動作は問題ございませんでした。

状態は写真をご確認いただきご理解の上ご購入をお願いいたします。
よろしくお願いいたします。
<Response [200]>
接続形式···有線
キースイッチ···静電容量無接点方式
キーボード配列···英語
使用していなかったので出品します。
日焼けあり
箱無し
USBケーブルあり
キートップはオプション品のカラーキートップに変えております。
<Response [200]>
※中古のため現品渡し＆ノークレーム・ノーリターンでお願いします。

HHKB Pro2
静電容量無接点方式キーボード
USB/AtoMiniケーブル、予備の純正チルト脚、キープラー付
機能に問題なし
キーキャップにテカリ有。
気になる方は脱脂して使用してください。
<Response [200]>
【プロ仕様・最高峰の信頼性】
ご覧いただきありがとうございます。
Lexar（レキサー）のプロフェッショナルシリーズ「SILVER PLUS」256GBの2個セット、新品未開封品です。
映像制作プロジェクト（8K/4K撮影）に向けて予備機材として準備しておりましたが、さらなる上位構成への変更に伴い、未使用のまま出品いたします。
最近メルカリ内で散見される「1TBで数千円」といった安価な粗悪品・偽物とは異なり、家電量販店等で取り扱われる正真正銘の国内正規品です。データの消失が許されないプロの現場や、大切な思い出の記録に安心

So, should probably have a like struct-adjacent thing for this, right? Or maybe a map. 
In my mind, the way that the "check for new items" thing goes is that we check the page every few minutes, and get the IDs for all the 120 items there. Then, we check "hey, are any of these not already in the map". If any aren't already in the map, then and only then do we need to request additional information for those items (namely, just request all their info using the `get_listing_json_from_id` fn).

I'm kinda completely inexperienced with python, so surely this is just like a dict with a bunch of different keys right? That like ~= a JSON object and, that's how I'd do this in JS.
I'm not gonna do this yet, because I'd like to get something Mercari-only working first, but some like something where we have like `MercariItem`, `JDirectItem`, `etcItem` that all have their own methods for this would be nice so I don't have to do like "if id matches this form, then from this marketplace..."

In [ ]:
for x in listings:
